In [1]:
import pandas as pd
#Load the file
df = pd.read_csv("parking_data.csv")

In [2]:
print(df.shape)


(2000, 15)


In [3]:
df.head()


,employee_id,site,year,parking_fee_chf,main_mode,uses_car,car_days_per_week,home_commune,home_postcode,distance_km,age,gender,n_children,fte,gross_income_chf
0,E1000,Lausanne,2024,2.0,car,1,2.2,Lausanne,1004,2.9,23,M,0,0.8,59300.0
1,E1000,Lausanne,2025,4.0,car,1,2.0,Lausanne,1004,2.9,23,M,0,0.8,59300.0
2,E1001,Lausanne,2024,2.0,car,1,3.5,Le Mont-sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0
3,E1001,Lausanne,2025,4.0,car,1,7.4,Le Mont-sur-Lausanne,1052,5.6,51,F,1,1.0,124300.0
4,E1002,Lausanne,2024,2.0,car,1,2.6,Chavannes-pres-Renens,1022,7.7,39,M,0,1.0,72800.0


In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   employee_id        2000 non-null   str    
 1   site               2000 non-null   str    
 2   year               2000 non-null   int64  
 3   parking_fee_chf    2000 non-null   float64
 4   main_mode          2000 non-null   str    
 5   uses_car           2000 non-null   int64  
 6   car_days_per_week  2000 non-null   float64
 7   home_commune       2000 non-null   str    
 8   home_postcode      2000 non-null   int64  
 9   distance_km        2000 non-null   float64
 10  age                2000 non-null   int64  
 11  gender             2000 non-null   str    
 12  n_children         2000 non-null   int64  
 13  fte                2000 non-null   float64
 14  gross_income_chf   1890 non-null   float64
dtypes: float64(5), int64(5), str(5)
memory usage: 234.5 KB


In [5]:
print("\nMissing values per column:")
print(df.isna().sum())
# missing values in the gross_income_chf column 


Missing values per column:
employee_id            0
site                   0
year                   0
parking_fee_chf        0
main_mode              0
uses_car               0
car_days_per_week      0
home_commune           0
home_postcode          0
distance_km            0
age                    0
gender                 0
n_children             0
fte                    0
gross_income_chf     110
dtype: int64


In [6]:
print("\nUnique values before cleaning:")
print(sorted(df["home_commune"].dropna().unique()))
print("Number of unique values:", df["home_commune"].nunique()) # Riduci nomi comuni (metti in minuscolo tutti i comuni)



Unique values before cleaning:
['ARDON', 'AYENT', 'Arbaz', 'Ardon', 'Ayent', 'BOTTENS', 'BUSSIGNY', 'Bottens', 'Bramois', 'Bussigny', 'CHAVANNES PRES RENENS', 'CRISSIER', 'CUGY', 'Chavannes-pres-Renens', 'Cheseaux-sur-Lausanne', 'Conthey', 'Corcelles-le-Jorat', 'Crissier', 'Cugy', 'ECUBLENS', 'EPALINGES', 'Ecublens', 'Epalinges', 'Evolene', 'FROIDEVILLE', 'Froideville', 'Grimisuat', 'Heremence', 'LAUSANNE', 'LE MONT SUR LAUSANNE', 'LUTRY', 'Lausanne', 'Le Mont-sur-Lausanne', 'Lutry', 'MEZIERES', 'MONTPREVEYRES', 'MORGES', 'Mezieres', 'Montpreveyres', 'Morges', 'Nendaz', 'PRILLY', 'PULLY', 'Prilly', 'Pully', 'RENENS', 'ROPRAZ', 'Renens', 'Romanel-sur-Lausanne', 'Ropraz', 'SAVIEZE', 'SAVIGNY', 'SION', 'Saint-Martin', 'Salins', 'Savieze', 'Savigny', 'Servion', 'Sion', 'Vetroz', 'Vex', 'Vulliens']
Number of unique values: 62


In [7]:
df["home_commune"] = (
    df["home_commune"]
    .astype("string")            # keeps NaN as <NA> instead of the string "nan"
    .str.strip()                 # remove leading/trailing spaces
    .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces
    .str.lower()                 # consistent lowercase
)
df["home_commune"] = df["home_commune"].str.title()

In [8]:
print("\nUnique values after cleaning:")
print(sorted(df["home_commune"].dropna().unique()))
print("Number of unique values:", df["home_commune"].nunique())


Unique values after cleaning:
['Arbaz', 'Ardon', 'Ayent', 'Bottens', 'Bramois', 'Bussigny', 'Chavannes Pres Renens', 'Chavannes-Pres-Renens', 'Cheseaux-Sur-Lausanne', 'Conthey', 'Corcelles-Le-Jorat', 'Crissier', 'Cugy', 'Ecublens', 'Epalinges', 'Evolene', 'Froideville', 'Grimisuat', 'Heremence', 'Lausanne', 'Le Mont Sur Lausanne', 'Le Mont-Sur-Lausanne', 'Lutry', 'Mezieres', 'Montpreveyres', 'Morges', 'Nendaz', 'Prilly', 'Pully', 'Renens', 'Romanel-Sur-Lausanne', 'Ropraz', 'Saint-Martin', 'Salins', 'Savieze', 'Savigny', 'Servion', 'Sion', 'Vetroz', 'Vex', 'Vulliens']
Number of unique values: 41


In [9]:
# Make sure income is numeric (non-numeric junk becomes NaN)
df["gross_income_chf"] = pd.to_numeric(df["gross_income_chf"], errors="coerce")

# 1. Sort so "previous" and "next" year are really adjacent rows per employee
df = df.sort_values(["employee_id", "year"]).reset_index(drop=True)

# 2. Check how many are missing before filling
missing_before = df["gross_income_chf"].isna().sum()
print("Missing before:", missing_before)

# 3. Keep track of which values were imputed (useful for robustness checks later)
df["income_imputed"] = df["gross_income_chf"].isna()

# 4. Fill within each employee: first from previous year, then from next year
df["gross_income_chf"] = (
    df.groupby("employee_id")["gross_income_chf"]
      .transform(lambda s: s.ffill().bfill())
)

# 5. Check the result
missing_after = df["gross_income_chf"].isna().sum()
print("Missing after:", missing_after)
print("Filled:", missing_before - missing_after)

# Employees where income is still missing in both years (can't be filled this way)
still_missing = df[df["gross_income_chf"].isna()]
print(still_missing[["employee_id", "year"]])

Missing before: 110
Missing after: 8
Filled: 102
     employee_id  year
82         E1041  2024
83         E1041  2025
1176       E1588  2024
1177       E1588  2025
1432       E1716  2024
1433       E1716  2025
1544       E1772  2024
1545       E1772  2025


In [11]:
# 8 values still missing (some employees have missing income in both years, so can't be filled this way) might have to drop these rows later if it's a necessary variable for analysis

In [12]:
df.to_csv("parking_data_clean.csv", index=False)